In [1]:
import torch
import torch.nn as nn
import esm
import csv
from datetime import datetime
import os
from transformers import BertTokenizer, BertModel
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
timestamp = datetime.now().strftime('%d-%b_%H-%M-%S')

c:\Users\brian\OneDrive\Documents\github\Mutation-Prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ProtBertClassifier(nn.Module):
    def __init__(self, num_labels=2, freeze_bert=False):
        super(ProtBertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("Rostlab/prot_bert")
        
        if freeze_bert:
            for param in self.bert[-5:].parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = (outputs.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1).unsqueeze(-1)
        # pooled_output = outputs.pooler_output 
        logits = self.classifier(pooled_output)
        return logits



In [3]:
class MutationDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

    def __len__(self):
        return len(self.sequences)


In [12]:
df = pd.read_csv("datasets/mutation_30mer_dataset153.csv")
patho_df = df[df["Label"] == 1]
benign_df = df[df["Label"] == 0]

min_count = min(len(patho_df), len(benign_df))

pathogenic_balanced = resample(patho_df, replace=False, n_samples=min_count, random_state=13)
benign_balanced = resample(benign_df, replace=False, n_samples=min_count, random_state=13)

balanced_df = pd.concat([pathogenic_balanced, benign_balanced]).sample(frac=1, random_state=13) 

In [13]:
def collate_fn_stringbatch(batch):
    """Batch is a list of (sequence, label) tuples."""
    seqs, labels = zip(*batch)

    # ProtBERT wants space-separated AA codes
    spaced = [' '.join(list(seq)) for seq in seqs]
    tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
    
    tokens = tokenizer(
        list(spaced),
        padding=True,
        truncation=True,
        max_length=64,            # 30-mer fits easily
        return_tensors="pt"
    )

    tokens["labels"] = torch.tensor(labels, dtype=torch.long)
    return tokens       


In [14]:
X = balanced_df["Mut_30mer"].values
y = balanced_df["Label"].values

xtrain, xtest, ytrain, ytest= train_test_split(X, y, test_size=0.15, stratify=y)

training_data = MutationDataset(xtrain, ytrain)
test_data = MutationDataset(xtest, ytest)
 
train_loader = DataLoader(training_data, batch_size=64, shuffle=True, collate_fn=collate_fn_stringbatch)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=collate_fn_stringbatch)


In [15]:
model = ProtBertClassifier().to(device)

In [16]:
model

ProtBertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30, 1024, padding_idx=0)
      (position_embeddings): Embedding(40000, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-29): 30 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,), eps=1

In [20]:
device

device(type='cpu')

In [ ]:
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

optimizer = torch.optim.SGD(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()
metrics_log = []
train_losses, val_losses = [], []
epochs = 45
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

for ep in range(1, epochs + 1):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = model(batch["input_ids"], batch["attention_mask"])

        loss = criterion(logits, batch["labels"])
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    print(f"Epoch {ep}: \tTrain Loss = {train_loss:.4f}, Train Accuracy = {train_acc:.4f}")
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            logits = model(batch["input_ids"], batch["attention_mask"])

            loss = criterion(logits, batch["labels"])

            test_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    val_acc = correct / total
    val_loss = test_loss / len(test_loader)
    val_losses.append(val_loss)
    metrics_log.append({
    "epoch": ep,
    "train_loss": train_loss,
    "train_acc": train_acc,
    "val_loss": val_loss,
    "val_acc": val_acc
    })
    scheduler.step(val_loss)
    print(f"\t \t Test Loss = {val_loss:.4f},  Test Accuracy = {val_acc:.4f}\n")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        # Optional: save best model here
        torch.save(model.state_dict(), f"model_{timestamp}.pt")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"⏹️ Early stopping triggered at epoch {ep}. No improvement for {patience} epochs.")
        break

KeyboardInterrupt: 

In [ ]:
folder = "custom_protbert"
os.makedirs(folder, exist_ok=True)

# Create CSV filename inside the folder

csv_log_path = os.path.join(folder, f"metrics_cproto_{timestamp}.csv")

# Now write the file
with open(csv_log_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
    writer.writeheader()
    writer.writerows(metrics_log)

print(f"✅ CSV saved to {csv_log_path}")

✅ CSV saved to custom_esm\metrics_cesm_20250501_015703.csv
